# Visual Health — embeddings EyePACS (validation externe)

2e validation externe de la rétinopathie diabétique : **EyePACS**
(compétition Kaggle « diabetic-retinopathy-detection », 35 126 images
annotées en 5 stades, programme de dépistage américain). Les mêmes
appareils y photographient malades et sains, contrairement à APTOS.

Ce notebook tire **2 500 patients au hasard** (graine 0, ~5 000 images),
n'extrait que leurs images, calcule les embeddings RETFound (`crop-mask`)
et zippe le résultat. L'évaluation se fait en local
(`scripts/external_validation.py --external eyepacs`).

**Avant de lancer (Colab)** :
1. Une seule fois : accepter le règlement de
   [diabetic-retinopathy-detection](https://www.kaggle.com/competitions/diabetic-retinopathy-detection)
   (`Late Submission` / `I understand and accept`).
2. `Exécution > Modifier le type d'exécution > GPU T4`.
3. Secrets 🔑 (accès activé) : `HF_TOKEN` et `KAGGLE_API_TOKEN` (clé `KGAT_…`).
4. `Exécution > Tout exécuter` (~40-60 min : téléchargement de ~35 Go,
   extraction, embeddings). Le zip se télécharge à la fin, puis
   **déconnecter et supprimer l'environnement d'exécution**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : choisir GPU T4"
!df -h /content | tail -1

In [ ]:
import os
WORK = "/content"
%cd {WORK}
![ -d visual-health-platform ] || git clone -q https://github.com/sumarc183-design/visual-health-platform.git
!cd visual-health-platform && git fetch -q && git reset -q --hard origin/main && git log --oneline -1
%cd {WORK}/visual-health-platform
!pip install -q -r requirements.txt
!pip install -q -U kaggle
!apt-get -qq install -y p7zip-full > /dev/null

In [ ]:
import glob
from google.colab import userdata

TARGET = "third_party/retfound/checkpoints/RETFound_mae_natureCFP.pth"
if not os.path.exists(TARGET):
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # jamais écrit dans le notebook
    !python scripts/download_retfound_checkpoint.py --repo-id YukunZhou/RETFound_mae_natureCFP --out third_party/retfound/checkpoints
CHECKPOINT = sorted(glob.glob("third_party/retfound/checkpoints/*.pth"))[0]

token = userdata.get("KAGGLE_API_TOKEN").strip()
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
    f.write(token)
os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)
os.environ["KAGGLE_API_TOKEN"] = token
print(CHECKPOINT)

## Échantillon de patients et extraction

Seules les images des 2 500 patients tirés sont extraites des 5 archives
d'entraînement ; les archives sont ensuite supprimées.

In [ ]:
import numpy as np
import pandas as pd

ROOT = f"{WORK}/eyepacs"
DL = f"{WORK}/eyepacs_zip"
os.makedirs(ROOT, exist_ok=True)
os.makedirs(DL, exist_ok=True)
COMP = "diabetic-retinopathy-detection"
if not os.path.exists(f"{ROOT}/trainLabels.csv"):
    !kaggle competitions download -c {COMP} -f trainLabels.csv.zip -p {DL} -q
    !unzip -q -o {DL}/trainLabels.csv.zip -d {ROOT}

labels = pd.read_csv(f"{ROOT}/trainLabels.csv")
patients = sorted(labels["image"].str.rsplit("_", n=1).str[0].unique())
chosen = set(np.random.default_rng(0).choice(patients, size=2500, replace=False))
names = labels.loc[labels["image"].str.rsplit("_", n=1).str[0].isin(chosen), "image"]
with open(f"{DL}/list.txt", "w") as f:
    f.write("\n".join(f"train/{n}.jpeg" for n in names))
print(len(chosen), "patients ;", len(names), "images ; stades", labels.set_index("image").loc[names, "level"].value_counts().sort_index().to_dict())

In [ ]:
if len(glob.glob(f"{ROOT}/train/*.jpeg")) < len(names):
    for part in range(1, 6):
        !kaggle competitions download -c {COMP} -f train.zip.00{part} -p {DL} -q
    !7z x {DL}/train.zip.001 -o{ROOT} -i@{DL}/list.txt -y -bso0 -bsp0
    !rm -f {DL}/train.zip.00*
print(len(glob.glob(f"{ROOT}/train/*.jpeg")), "images extraites")
!df -h /content | tail -1

## Embeddings RETFound

Même extraction que `run_baseline.py` ; `eyepacs_labels.csv` contient la
clé, le stade, le patient et le format d'image d'origine.

In [ ]:
import time

t0 = time.time()
!python -u scripts/extract_embeddings.py --dataset eyepacs --data-root {ROOT} --checkpoint {CHECKPOINT} \
    --out-dir embeddings --batch-size 64 --num-workers 2 2>&1 | grep -v -i warning
out = "embeddings/eyepacs_crop-mask_224_RETFound_mae_natureCFP.npz"
if not os.path.exists(out) or os.path.getmtime(out) < t0:
    raise RuntimeError("extract_embeddings.py a échoué (voir la sortie ci-dessus)")

In [ ]:
import zipfile

archive = f"{WORK}/visual_health_eyepacs_embeddings.zip"
with zipfile.ZipFile(archive, "w") as z:
    for f in (out, "embeddings/eyepacs_labels.csv"):
        z.write(f)
print("->", archive, f"({os.path.getsize(archive) / 1e6:.0f} Mo)")
from google.colab import files
files.download(archive)